# Signal study: do the Consumer-Trend Score and the bottleneck company score predict anything?

Section 11.4 of the build spec. **Point-in-time discipline:** each score uses only data dated on or before the score date; forward returns use closes strictly after it. From Phase 11 the same question is asked of the bottleneck company score (section 4A.7) with the same rule, and the two are reported separately and never blended. Whatever the result, it is reported honestly; a null or insufficient-data result is the expected outcome on a synthetic corpus.

_Notebook generated by `scripts/build_signal_notebook.py` on 2026-11-08; re-run that script to refresh the outputs._

In [1]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path
exec(Path('../evals/signal_study.py').read_text())  # defines run_study()
from IPython.display import Markdown
Markdown(run_study())

## Signal study: point-in-time Consumer-Trend Score vs forward returns

_Run 2026-11-08 on the fixture corpus (synthetic social data, real closes for 10 tickers recorded 2026-10-27)._

- Score dates: 2026-09-15, 2026-09-22, 2026-09-29 (weekly, point-in-time).
- Investable trend signals produced: 63; with a price fixture: 27 (8 distinct tickers).

| Horizon | Top-tercile mean excess return vs SPY (95% CI) | Bottom-tercile | n top / n bottom |
|---|---|---|---|
| 30d | +9.05% [+4.59%, +13.51%] | +0.84% [-7.24%, +8.93%] | 8 / 5 |
| 60d | n/a | n/a | 0 / 0 |
| 90d | n/a | n/a | 0 / 0 |

Per-observation detail:

| as of | entity | ticker | score | 30d | 60d | 90d |
|---|---|---|---|---|---|---|
| 2026-09-29 | Starbucks | SBUX | 23 | n/a | n/a | n/a |
| 2026-09-22 | Chipotle | CMG | 26 | -13.97% | n/a | n/a |
| 2026-09-29 | Chipotle | CMG | 27 | n/a | n/a | n/a |
| 2026-09-15 | Xbox Game Pass | MSFT | 28 | +10.02% | n/a | n/a |
| 2026-09-29 | Xbox Game Pass | MSFT | 28 | n/a | n/a | n/a |
| 2026-09-22 | Starbucks | SBUX | 29 | +8.25% | n/a | n/a |
| 2026-09-22 | Xbox Game Pass | MSFT | 29 | +4.09% | n/a | n/a |
| 2026-09-15 | Starbucks | SBUX | 31 | -0.47% | n/a | n/a |
| 2026-09-29 | Nike | NKE | 31 | n/a | n/a | n/a |
| 2026-09-22 | Nike | NKE | 32 | +6.30% | n/a | n/a |
| 2026-09-29 | AirPods | AAPL | 32 | n/a | n/a | n/a |
| 2026-09-22 | Kirkland Signature | COST | 36 | +12.55% | n/a | n/a |
| 2026-09-29 | Costco | COST | 39 | n/a | n/a | n/a |
| 2026-09-22 | Costco | COST | 42 | +12.55% | n/a | n/a |
| 2026-09-15 | AirPods | AAPL | 43 | +2.88% | n/a | n/a |
| 2026-09-29 | HOKA | DECK | 44 | n/a | n/a | n/a |
| 2026-09-15 | Chipotle | CMG | 44 | -9.20% | n/a | n/a |
| 2026-09-29 | Kirkland Signature | COST | 45 | n/a | n/a | n/a |
| 2026-09-22 | e.l.f. | ELF | 49 | +14.10% | n/a | n/a |
| 2026-09-15 | Nike | NKE | 49 | +3.80% | n/a | n/a |
| 2026-09-22 | AirPods | AAPL | 50 | +3.32% | n/a | n/a |
| 2026-09-22 | HOKA | DECK | 50 | +8.51% | n/a | n/a |
| 2026-09-15 | Kirkland Signature | COST | 52 | +9.43% | n/a | n/a |
| 2026-09-15 | Costco | COST | 54 | +9.43% | n/a | n/a |
| 2026-09-29 | e.l.f. | ELF | 56 | n/a | n/a | n/a |
| 2026-09-15 | e.l.f. | ELF | 63 | +24.30% | n/a | n/a |
| 2026-09-15 | HOKA | DECK | 69 | +11.06% | n/a | n/a |

## Signal study: point-in-time bottleneck company score (Phase 11, reported separately)

- Curated supplier scores produced: 15 over 3 dates (5 tickers: CSCO, CVX, HON, UNP, XOM); with a price fixture: 0.

**Result: insufficient data; no predictive claim is made.** The offline bundle carries closes for the ten demo tickers and none of them is a curated bottleneck supplier (HON, CSCO, XOM, CVX, UNP), so no forward return can be paired with a bottleneck score yet. The synthetic filings corpus would make any such pairing meaningless anyway; the table above shows the method, the live snapshots in `data/bottleneck_snapshots.sqlite` (180-day retention) are what this study will read once they exist.

### Caveats (read these before the numbers)
- **Synthetic social data.** The fixture corpus was generated with designed trend shapes; any relationship with real price moves is coincidence by construction.
- **Survivorship / selection.** Only tickers with a bundled price fixture are evaluated.
- **Tiny sample, multiple horizons.** Three horizons × two groups on a handful of names: treat every interval above as uninformative.
- **Forward windows truncated** by the 90-day close history; later horizons have fewer observations.
- **Bottleneck score (Phase 11):** scored on the same synthetic corpus; the layer z-scores depend on designed fixture dates, and the curated suppliers have no bundled closes, so the second study is a method demonstration only until `jobs/refresh_bottlenecks.py` has run live for two quarters.
- **What would make this real:** run `jobs/refresh_trends.py` daily in live mode for 6+ months, persist each snapshot, then re-run this study on stored scores vs realised returns, with the same point-in-time rule and a pre-registered hypothesis (top-decile 60-day excess return > 0).


## How to turn this into a real test

1. Run `python -m options_scout.jobs.refresh_trends --mode live` and `python -m options_scout.jobs.refresh_bottlenecks --mode live` daily; every snapshot is stored with its `as_of` (trend snapshots for 48 hours, bottleneck snapshots for 180 days, since they hold no personal data).
2. After 6+ months, join stored scores to realised 30/60/90-day returns (MarketData.app candles) and SPY, one study per signal.
3. Pre-register the hypothesis (top-decile 60-day excess return > 0), report the CI, the sample size, and the number of hypotheses tested. Survivorship bias: keep delisted names in the sample.
